# Experiment 1 — Device × Model Profiling

**Objective:** Profile YOLO26 (nano / small / medium) in tracking mode at 640×640
on all three MOT17 sequences (sparse / moderate / dense).  Collect throughput,
latency, memory, and tracking-quality metrics to produce Table I of the paper.

**Hypothesis:** (from methodology) Track continuity metrics will degrade faster
than detection-count metrics under increasing pedestrian density.

**Success criteria:**
- All 9 raw CSVs present in `results/raw/` with the correct row count per sequence.
- Table I DataFrame renders with finite values for every cell.

---

**Device-specific notes:**
- RPi 5: set `DEVICE = "cpu"`, use `.pt` models.
- Jetson devices: export TensorRT engines first (`model.export(format='engine')`),
  then point `MODEL_VARIANTS` at the `.engine` files and set `DEVICE = "cuda:0"`.
- Development machine: `DEVICE` is auto-detected below.

In [ ]:
# Device profile — controls which models, device, and backend are used.
# On edge hardware, set the DEVICE_PROFILE env var before launching Jupyter:
#
#   DEVICE_PROFILE=rpi5_hailo.yaml jupyter lab
#   DEVICE_PROFILE=jetson_nano.yaml jupyter lab
#
# Without the env var the desktop fallback is used automatically.
import os
from benchmark.device_profile import load_profile
from benchmark.config import DATA_ROOT, RESULTS_RAW, SEQUENCES, SEQ_SUFFIX, IMGSZ_BASE

PROFILE = load_profile(os.environ.get("DEVICE_PROFILE"))

print(f"Device  : {PROFILE.device_label}")
print(f"Backend : {PROFILE.backend}  |  torch device: {PROFILE.torch_device}")
print(f"Models  : {PROFILE.model_variants}")
print(f"Tag     : {PROFILE.result_tag}")

# Set to True to skip sequences whose CSV already exists (safe to re-run)
SKIP_EXISTING = True

In [ ]:
# Inference loop: all profile model variants × 3 sequences at 640×640
# Hailo backend uses run_sequence_hailo(); all others use run_sequence() via ultralytics.
# Failed model loads are reported and skipped — the loop continues to the next variant.
from benchmark.device_profile import try_load_model, resolve_model_path
from benchmark.runner import run_sequence

if PROFILE.backend == "hailo":
    from benchmark.hailo_runner import run_sequence_hailo

failed_loads = []

for model_path in PROFILE.model_variants:
    print(f"\n── Model: {model_path} ──")
    resolved = resolve_model_path(model_path)

    for seq_name in SEQUENCES:
        out_csv = RESULTS_RAW / f"{seq_name}_{model_path}_{IMGSZ_BASE}_{PROFILE.result_tag}.csv"

        if SKIP_EXISTING and out_csv.exists():
            print(f"  {seq_name}: skip (CSV exists)")
            continue

        seq_dir = DATA_ROOT / f"{seq_name}-{SEQ_SUFFIX}"
        print(f"  {seq_name}: running ...", end=" ", flush=True)

        try:
            if PROFILE.backend == "hailo":
                df = run_sequence_hailo(resolved, seq_dir, imgsz=IMGSZ_BASE, out_csv=out_csv)
            else:
                model, err = try_load_model(model_path, PROFILE.torch_device)
                if err:
                    print(f"  SKIP — could not load: {err}")
                    failed_loads.append((model_path, err))
                    break   # skip remaining sequences for this variant
                df = run_sequence(model, seq_dir, imgsz=IMGSZ_BASE, out_csv=out_csv)
                del model
        except Exception as exc:
            print(f"  FAILED: {exc}")
            failed_loads.append((model_path, str(exc)))
            break

        print(f"done — {len(df)} frames, mean {df['n_detections'].mean():.1f} det/frame")

if failed_loads:
    print("\n── Load/run failures ──")
    for name, reason in failed_loads:
        print(f"  {name}: {reason}")

In [ ]:
# Timing and memory summary from raw CSVs
import numpy as np
import pandas as pd
from benchmark.config import WARMUP_FRAMES

timing_rows = []
for model_path in PROFILE.model_variants:
    for seq_name in SEQUENCES:
        csv_path = RESULTS_RAW / f"{seq_name}_{model_path}_{IMGSZ_BASE}_{PROFILE.result_tag}.csv"
        if not csv_path.exists():
            continue
        df = pd.read_csv(csv_path)

        # Exclude warm-up NaN rows from timing statistics
        t = df["inference_ms"].dropna()
        timing_rows.append({
            "model":        model_path,
            "seq":          seq_name,
            "fps":          round(1000 / t.median(), 1),
            "median_ms":    round(t.median(), 2),
            "p95_ms":       round(t.quantile(0.95), 2),
            "peak_mem_mb":  round(df["mem_bytes"].max() / 1e6, 1),
            "mean_det":     round(df["n_detections"].mean(), 1),
        })

timing_df = pd.DataFrame(timing_rows)
print(timing_df.to_string(index=False))

In [ ]:
# MOT tracking-quality metrics per (model, sequence)
from benchmark.metrics import compute_mot_metrics

mot_rows = []
for model_path in PROFILE.model_variants:
    for seq_name in SEQUENCES:
        csv_path = RESULTS_RAW / f"{seq_name}_{model_path}_{IMGSZ_BASE}_{PROFILE.result_tag}.csv"
        if not csv_path.exists():
            continue
        seq_dir = DATA_ROOT / f"{seq_name}-{SEQ_SUFFIX}"
        print(f"  Computing metrics: {model_path} / {seq_name} ...", end=" ", flush=True)
        m = compute_mot_metrics(csv_path, seq_dir)
        print(f"MOTA={m['mota']:.3f}  IDF1={m['idf1']:.3f}  IDSW={m['num_switches']}  IDSW/GT={m['idsw_per_gt_track']:.3f}")
        mot_rows.append({"model": model_path, "seq": seq_name, **m})

mot_df = pd.DataFrame(mot_rows)

In [ ]:
# Table I — merged profiling table (timing + tracking quality)
# IDSW/GT-track is the normalised identity confusion signal from methodology v2.
table1 = timing_df.merge(
    mot_df[["model","seq","mota","idf1","num_switches","idsw_per_gt_track","frag_ratio"]],
    on=["model","seq"]
)

# Round for display
for col in ["mota","idf1","idsw_per_gt_track","frag_ratio"]:
    table1[col] = table1[col].round(3)

table1 = table1.set_index(["model","seq"])
print(f"\nTable I — {PROFILE.device_label} Profiling")
print(table1.to_string())

# Save tagged CSV so results from each device land in separate files
table1_path = RESULTS_RAW.parent / "profiling" / f"table1_profiling_{PROFILE.result_tag}.csv"
table1_path.parent.mkdir(parents=True, exist_ok=True)
table1.to_csv(table1_path)
print(f"\nSaved to {table1_path}")

In [ ]:
# Model selection for Experiment 2:
# Highest FPS among variants with non-trivial tracking (mean_det > 1 across all sequences)
# valid = timing_df.groupby("model").filter(lambda g: (g["mean_det"] > 1).all())
# best  = valid.groupby("model")["fps"].mean().idxmax()

# print(f"\nSelected model for Experiment 2: {best}")
# print("Justification: highest mean FPS across sequences with non-trivial track output.")
# print("\n>>> Set SELECTED_MODEL = '" + best + "' in notebook 02.")

## Results

- Key observations: *(fill after run)*
- Surprises: *(e.g., unexpected IDSW spike on dense sequence)*
- Selected model for Experiment 2: *(from cell above)*

## Next steps

- Copy `SELECTED_MODEL` value into `notebooks/02_experiment2_resolution.ipynb`.
- Repeat this notebook on each edge device; copy raw CSVs to `results/raw/` on the dev machine.
- Run `notebooks/03_results_figures.ipynb` to generate paper figures from all collected CSVs.